In [1]:
from sqlalchemy import create_engine, text
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [3]:
engine = create_engine(f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}")

xls = pd.ExcelFile("C:\\Users\\sniks\\OneDrive\\Desktop\\dbms\\project\\data\\Supply chain logisitcs problem.xlsx")

orders = pd.read_excel(xls, "OrderList")
freight = pd.read_excel(xls, "FreightRates")
wh_costs = pd.read_excel(xls, "WhCosts")
plant_ports = pd.read_excel(xls, "PlantPorts")
vmi = pd.read_excel(xls, "VmiCustomers")

def clean_columns(df):
    df.columns = (df.columns.str.lower().str.strip().str.replace(" ", "_").str.replace("/", "_").str.replace("-", "_"))
    return df


orders = clean_columns(orders)
freight = clean_columns(freight)
wh_costs = clean_columns(wh_costs)
plant_ports = clean_columns(plant_ports)
vmi = clean_columns(vmi)


orders.to_sql("orders", engine, schema="supply_chain", if_exists="replace", index=False)
freight.to_sql("freight_rates", engine, schema="supply_chain", if_exists="replace", index=False)
wh_costs.to_sql("warehouse_costs", engine, schema="supply_chain", if_exists="replace", index=False)
plant_ports.to_sql("plant_ports", engine, schema="supply_chain", if_exists="replace", index=False)
vmi.to_sql("vmi_customers", engine, schema="supply_chain", if_exists="replace", index=False)

14

In [4]:
"""Orders by Origin Port"""

origin_port_summary = (orders.groupby("origin_port").agg(total_orders=("order_id", "count"),avg_weight=("weight", "mean")).reset_index().sort_values(by="total_orders", ascending=False))

origin_port_summary["avg_weight"] = origin_port_summary["avg_weight"].round(2)
origin_port_summary


,origin_port,total_orders,avg_weight
0,PORT04,9041,16.70
2,PORT09,173,185.94
1,PORT05,1,2.10


##  Generated tabular summaries to identify high-activity origin ports, highlighting key shipment hubs and so watching out that PORT04 has been the best one so far and the others need equivalent work for improvments.


In [5]:
"""Domestic vs International Shipments"""
orders["shipment_type"] = orders.apply(lambda x: "Domestic" if x["origin_port"] == x["destination_port"] else "International",axis=1)

geo_distribution = (orders.groupby("shipment_type").agg(total_orders=("order_id", "count"),avg_weight=("weight", "mean")).reset_index())

geo_distribution


,shipment_type,total_orders,avg_weight
0,Domestic,173,185.942265
1,International,9042,16.694270


##  Compared domestic and international shipments to uncover differences in volume and delivery behavior making us realise to pay more attention to domestic customers


In [6]:
"""Branches with Most Failed Deliveries"""
failed_deliveries = (orders[orders["ship_late_day_count"] > 0].groupby("plant_code").size().reset_index(name="failed_orders").sort_values(by="failed_orders", ascending=False))

failed_deliveries


,plant_code,failed_orders
0,PLANT03,174
1,PLANT08,18


##  Analyzed branch-level delivery failures to surface operational bottlenecks and thus concluding that PLANT03 needs potential improvement.


In [7]:
"""Shipment Outlier Analysis (Weight-Based)"""
mean_weight = orders["weight"].mean()
std_weight = orders["weight"].std()

outliers = orders[orders["weight"] > (mean_weight + 2 * std_weight)]

outliers[["order_id", "weight"]]


,order_id,weight
17,1.447139e+09,181.700000
18,1.447309e+09,227.200000
21,1.447352e+09,265.100000
22,1.447212e+09,267.100000
23,1.447233e+09,271.100000
...,...,...
8038,1.447387e+09,413.770283
8041,1.447339e+09,209.439363
8042,1.447361e+09,212.439363
8050,1.447361e+09,161.568425


## • Identified shipment weight outliers that may increase handling complexity and logistics cost.


In [8]:
"""Shipment Charge Outlier Analysis"""
mean_charge = freight["rate"].mean()
std_charge = freight["rate"].std()

outliers = freight[freight["rate"] > (mean_charge + 2 * std_charge)]

outliers


,carrier,orig_port_cd,dest_port_cd,minm_wgh_qty,max_wgh_qty,svc_cd,minimum_cost,rate,mode_dsc,tpt_day_cnt,carrier_type
326,V444_4,PORT10,PORT09,41.281408,41.730464,DTD,7.8952,12.554687,AIR,3,V888888883_1
327,V444_4,PORT10,PORT09,41.735000,42.184056,DTD,7.9280,12.626999,AIR,3,V888888883_1
334,V444_4,PORT10,PORT09,44.910144,45.359200,DTD,9.1576,12.133181,AIR,3,V888888883_1
337,V444_4,PORT10,PORT09,46.270920,46.719976,DTD,9.2788,12.400381,AIR,3,V888888883_1
339,V444_4,PORT10,PORT09,47.178104,47.627160,DTD,9.3596,12.578515,AIR,3,V888888883_1
341,V444_4,PORT10,PORT09,48.085288,48.534344,DTD,9.4404,13.756648,AIR,3,V888888883_1
344,V444_4,PORT10,PORT09,49.446064,49.895120,DTD,9.5616,15.023849,AIR,3,V888888883_1
345,V444_4,PORT10,PORT09,49.899656,50.348712,DTD,9.6020,13.112916,AIR,3,V888888883_1
346,V444_4,PORT10,PORT09,50.353248,50.802304,DTD,9.6424,14.201983,AIR,3,V888888883_1
347,V444_4,PORT10,PORT09,50.806840,51.255896,DTD,9.6828,13.291049,AIR,3,V888888883_1


## • Detected abnormal freight charges through statistical inspection of shipment cost distributions, indicating most profitable customers 


In [9]:
"""Customers with Late Shipments"""
customer_late_summary = (orders.groupby("customer").agg(total_orders=("order_id", "count"),late_shipments=("ship_late_day_count", lambda x: (x > 0).sum())).reset_index())

customer_late_summary


,customer,total_orders,late_shipments
0,V555555555555555555_17,117,0
1,V555555555555555555_42,30,0
2,V555555555555555555_45,198,0
3,V555555555555555555_46,3,0
4,V555555555555555_23,1,0
5,V555555555555555_29,385,0
6,V555555555555555_44,9,9
7,V55555555555555_8,976,0
8,V5555555555555_16,12,0
9,V555555555555_31,45,0


##  Evaluated customer-wise late delivery patterns to identify reliability risks and watching out for loyal customers.


In [10]:
"""High Quantity but Low Shipment Volume Customers"""
customer_summary = (orders.groupby("customer").agg(shipment_count=("order_id", "count"),total_quantity=("unit_quantity", "sum")).reset_index())

high_value_low_volume = customer_summary[(customer_summary["shipment_count"] < customer_summary["shipment_count"].median()) &(customer_summary["total_quantity"] > customer_summary["total_quantity"].median())]

high_value_low_volume


,customer,shipment_count,total_quantity
1,V555555555555555555_42,30,470632
9,V555555555555_31,45,435868
38,V5555_36,29,931678
41,V555_41,11,201107


## Customers that are exploiting services, not ordering much, but weight based products are peaking in value.

In [11]:
"""customer segmentation"""
def segment_customer(cnt):
    if cnt > 100:
        return "VIP"
    elif cnt > 50:
        return "Premium"
    elif cnt > 10:
        return "Regular"
    else:
        return "New"

customer_summary["segment"] = customer_summary["shipment_count"].apply(segment_customer)
customer_summary


,customer,shipment_count,total_quantity,segment
0,V555555555555555555_17,117,266457,VIP
1,V555555555555555555_42,30,470632,Regular
2,V555555555555555555_45,198,116136,VIP
3,V555555555555555555_46,3,12080,New
4,V555555555555555_23,1,375,New
5,V555555555555555_29,385,1054980,VIP
6,V555555555555555_44,9,15125,New
7,V55555555555555_8,976,877824,VIP
8,V5555555555555_16,12,3434,Regular
9,V555555555555_31,45,435868,Regular


##  Segmented customers based on shipment frequency and quantity for behavioral insights into new, VIP and Regular customers. Offers can be made to New customers, and Discounts can be given to old customers to retain them.


In [12]:
"""churn prediction"""
last_activity = (orders.groupby("customer")["order_date"].max().reset_index())

latest_date = orders["order_date"].max()

last_activity["days_inactive"] = (latest_date - last_activity["order_date"]).dt.days

last_activity["status"] = last_activity["days_inactive"].apply(
    lambda x: "Active" if x < 30 else "At Risk" if x < 90 else "Legacy"
)

last_activity


,customer,order_date,days_inactive,status
0,V555555555555555555_17,2013-05-26,0,Active
1,V555555555555555555_42,2013-05-26,0,Active
2,V555555555555555555_45,2013-05-26,0,Active
3,V555555555555555555_46,2013-05-26,0,Active
4,V555555555555555_23,2013-05-26,0,Active
5,V555555555555555_29,2013-05-26,0,Active
6,V555555555555555_44,2013-05-26,0,Active
7,V55555555555555_8,2013-05-26,0,Active
8,V5555555555555_16,2013-05-26,0,Active
9,V555555555555_31,2013-05-26,0,Active


## All active customers, likely because membership date hasnot been included.

In [13]:
"""career productivity ranking"""
carrier_productivity = (orders.groupby("carrier").agg(shipments_handled=("order_id", "count"),avg_weight=("weight", "mean"),late_shipments=("ship_late_day_count", lambda x: (x > 0).sum())).reset_index())

carrier_productivity["success_rate"] = (1 - carrier_productivity["late_shipments"] / carrier_productivity["shipments_handled"])

carrier_productivity.sort_values(by="success_rate", ascending=False)


,carrier,shipments_handled,avg_weight,late_shipments,success_rate
2,V44_3,854,41.103810,0,1.000000
1,V444_1,2097,12.672628,9,0.995708
0,V444_0,6264,19.387044,183,0.970785


## Ranked logistics carriers using performance metrics derived from delivery outcomes and concluded that V44_3 is the best among all.


In [14]:
"""data quality check"""
data_quality = {"missing_origin_port": orders["origin_port"].isnull().sum(),
    "missing_destination_port": orders["destination_port"].isnull().sum(),
    "invalid_weight": (orders["weight"] <= 0).sum(),
    "missing_customer": orders["customer"].isnull().sum()
}

pd.DataFrame.from_dict(data_quality, orient="index", columns=["issue_count"])


,issue_count
missing_origin_port,0
missing_destination_port,0
invalid_weight,2
missing_customer,0


## Performed data quality checks to assess completeness and consistency before downstream analysis.
